In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import sys
from neuralop.models import TFNO
from neuralop import Trainer
from neuralop.training import AdamW
from neuralop.utils import count_model_params, get_project_root
from neuralop import LpLoss, H1Loss
from pathlib import Path

from diff_pde import load_diff_pde

device = "cuda"

n_train = 85000
n_val = 1000

save_path = Path("diffusion_data") / f"train{n_train}_test{n_val}"

train_loader, val_loader, data_processor = load_diff_pde(
    path=save_path,
    batch_size=64,
    test_batch_size=n_val,
)

val_loaders = {}
val_loaders[100] = val_loader

model = TFNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
)
model = model.to(device)

# Count and display the number of parameters
n_params = count_model_params(model)
print(f"\nOur model has {n_params} parameters.")
sys.stdout.flush()

optimizer = AdamW(
    model.parameters(), lr=1e-3, weight_decay=1e-4
)  # weight_decay can be tuned
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=1000, eta_min=1e-8
)

l2loss = LpLoss(d=2, p=2)  # L2 loss for function values
h1loss = H1Loss(d=2)  # H1 loss includes gradient information

train_loss = h1loss
eval_losses = {"h1": h1loss, "l2": l2loss}

print("\n### MODEL ###\n", model)
print("\n### OPTIMIZER ###\n", optimizer)
print("\n### SCHEDULER ###\n", scheduler)
print("\n### LOSSES ###")
print(f"\n * Train: {train_loss}")
print(f"\n * Test: {eval_losses}")
sys.stdout.flush()

trainer = Trainer(
    model=model,
    n_epochs=1000,
    device=device,
    data_processor=data_processor().to(device),
    wandb_log=True,  # Disable Weights & Biases logging for this tutorial
    eval_interval=20,  # Evaluate every 5 epochs
    use_distributed=False,  # Single GPU/CPU training
    verbose=True,  # Print training progress
)

trainer.train(
    train_loader=train_loader,
    test_loaders=val_loaders,
    optimizer=optimizer,
    scheduler=scheduler,
    regularizer=False,
    training_loss=train_loss,
    eval_losses=eval_losses,
)

In [ ]:
save_dir = Path("checkpoints")
save_dir.mkdir(exist_ok=True)

model_path = save_dir / "tfno_diff_pde.pt"
torch.save(model.state_dict(), model_path)

print(f"Model saved to {model_path}")

In [ ]:
# Testing with randomly drawn parameters a

import numpy as np
import torch
import matplotlib.pyplot as plt
from mpmath import zeta
from fenics import *

# Suppress FEniCS output
set_log_level(30)


def build_a_field_on_grid(y, X1, X2):
    # y: (N, L) numpy array
    # X1, X2: (G, G) numpy grids
    mu = 2
    c = float(0.9 / zeta(2))

    def k(j):
        return int(np.floor(-0.5 + np.sqrt(0.25 + 2 * j)))

    def m1(j):
        kj = k(j)
        return j - kj * (kj + 1) // 2

    def m2(j):
        return k(j) - m1(j)

    N, L = y.shape
    G = X1.shape[0]

    a = np.ones((N, G, G), dtype=np.float64)

    for j in range(L):
        j1 = j + 1
        freq1 = m1(j1)
        freq2 = m2(j1)
        psi = (
            c
            * (j1 ** (-mu))
            * np.cos(2 * np.pi * freq1 * X1)
            * np.cos(2 * np.pi * freq2 * X2)
        )
        a += y[:, j][:, None, None] * psi[None, :, :]

    return a  # (N, G, G)


def solve_pde_u_on_grid(y, x1_grid, x2_grid, nx=50, ny=50):
    mu = 2
    c = float(0.9 / zeta(2))

    def k(j):
        return int(np.floor(-0.5 + np.sqrt(0.25 + 2 * j)))

    def m1(j):
        kj = k(j)
        return j - kj * (kj + 1) // 2

    def m2(j):
        return k(j) - m1(j)

    mesh = UnitSquareMesh(nx, ny)
    V = FunctionSpace(mesh, "P", 1)

    bc = DirichletBC(V, Constant(0.0), "on_boundary")

    class CoefficientA(UserExpression):
        def __init__(self, y_coeffs, c, mu, **kwargs):
            super().__init__(**kwargs)
            self.y = y_coeffs
            self.c = c
            self.mu = mu

        def eval(self, values, x):
            x1 = x[0]
            x2 = x[1]
            a_val = 1.0
            for j in range(len(self.y)):
                j1 = j + 1
                freq1 = m1(j1)
                freq2 = m2(j1)
                psi_j = (
                    self.c
                    * (j1 ** (-self.mu))
                    * np.cos(2 * np.pi * freq1 * x1)
                    * np.cos(2 * np.pi * freq2 * x2)
                )
                a_val += self.y[j] * psi_j
            values[0] = a_val

        def value_shape(self):
            return ()

    f = Constant(1.0)
    u = TrialFunction(V)
    v = TestFunction(V)

    a_expr = CoefficientA(y, c, mu, degree=3)
    a_form = dot(a_expr * grad(u), grad(v)) * dx

    u_sol = Function(V)
    solve(a_form == f * v * dx, u_sol, bc)

    # Evaluate u on uniform grid (Gx1 x Gx2)
    G1 = len(x1_grid)
    G2 = len(x2_grid)
    U = np.zeros((G1, G2), dtype=np.float64)
    for i, x1 in enumerate(x1_grid):
        for j, x2 in enumerate(x2_grid):
            U[i, j] = u_sol(Point(float(x1), float(x2)))

    return U  # (G1, G2)


torch.set_grad_enabled(False)

device = "cuda:1"

N_eval = 10**1
L = 20
G = 100

# Uniform grid for evaluation and input fields
x1_grid = np.linspace(0.0, 1.0, G)
x2_grid = np.linspace(0.0, 1.0, G)
X1, X2 = np.meshgrid(x1_grid, x2_grid, indexing="ij")  # (G, G)

# Random parameters y in [-1,1]^20
y = (2.0 * np.random.rand(N_eval, L) - 1.0).astype(np.float64)

# Build input a(x1,x2;y) on uniform grid (reference input to the NN)
a_field = build_a_field_on_grid(y, X1, X2)  # (N, G, G)
a_in = torch.from_numpy(a_field).float().unsqueeze(1).to(device)  # (N,1,G,G)

# Reference u(x1,x2;y) via FEniCS
u_ref_list = []
for n in range(N_eval):
    u_ref_list.append(solve_pde_u_on_grid(y[n, :], x1_grid, x2_grid, nx=50, ny=50))
u_ref = (
    torch.from_numpy(np.stack(u_ref_list, axis=0)).float().unsqueeze(1).to(device)
)  # (N,1,G,G)

# Load model and predict
from neuralop.models import TFNO

model = TFNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
).to(device)

model.load_state_dict(torch.load("checkpoints/tfno_diff_pde.pt", map_location=device))
model.eval()
errs = []

bs = 4

for i in range(0, a_in.shape[0], bs):
    a_batch = a_in[i : i + bs]
    u_ref_b = u_ref[i : i + bs]
    u_pred_b = model(a_batch)
    err_b = torch.linalg.vector_norm(
        (u_ref_b - u_pred_b).reshape(u_ref_b.shape[0], -1), dim=1
    ) / torch.linalg.vector_norm(u_ref_b.reshape(u_ref_b.shape[0], -1), dim=1)
    errs.append(err_b.detach().cpu())

err = torch.cat(errs, dim=0)
err_np = err.numpy()

plt.figure(figsize=(8, 2.5))
plt.boxplot(
    err_np,
    vert=False,
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightblue", color="blue"),
    medianprops=dict(color="darkblue"),
    whiskerprops=dict(color="blue"),
    capprops=dict(color="blue"),
    flierprops=dict(
        markerfacecolor="blue",
        marker="o",
        markersize=3,
        linestyle="none",
        markeredgecolor="none",
    ),
)
plt.xscale("log")
plt.xlabel("Relative $L^2$ error (log scale)", fontsize=12)
plt.title(
    "Boxplot of relative $L^2$ Errors for the 2d parametric diffusion equation",
    fontsize=13,
)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# Use this if you have a precomputed set of parameters y for comparability

import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpmath import zeta
from fenics import *

set_log_level(30)


def build_a_field_on_grid(y, X1, X2):
    mu = 2
    c = float(0.9 / zeta(2))

    def k(j):
        return int(np.floor(-0.5 + np.sqrt(0.25 + 2 * j)))

    def m1(j):
        kj = k(j)
        return j - kj * (kj + 1) // 2

    def m2(j):
        return k(j) - m1(j)

    N, L = y.shape
    G = X1.shape[0]

    a = np.ones((N, G, G), dtype=np.float64)

    for j in range(L):
        j1 = j + 1
        freq1 = m1(j1)
        freq2 = m2(j1)
        psi = (
            c
            * (j1 ** (-mu))
            * np.cos(2 * np.pi * freq1 * X1)
            * np.cos(2 * np.pi * freq2 * X2)
        )
        a += y[:, j][:, None, None] * psi[None, :, :]

    return a


def solve_pde_u_on_grid(y, x1_grid, x2_grid, nx=50, ny=50):
    mu = 2
    c = float(0.9 / zeta(2))

    def k(j):
        return int(np.floor(-0.5 + np.sqrt(0.25 + 2 * j)))

    def m1(j):
        kj = k(j)
        return j - kj * (kj + 1) // 2

    def m2(j):
        return k(j) - m1(j)

    mesh = UnitSquareMesh(nx, ny)
    V = FunctionSpace(mesh, "P", 1)

    bc = DirichletBC(V, Constant(0.0), "on_boundary")

    class CoefficientA(UserExpression):
        def __init__(self, y_coeffs, c, mu, **kwargs):
            super().__init__(**kwargs)
            self.y = y_coeffs
            self.c = c
            self.mu = mu

        def eval(self, values, x):
            x1 = x[0]
            x2 = x[1]
            a_val = 1.0
            for j in range(len(self.y)):
                j1 = j + 1
                freq1 = m1(j1)
                freq2 = m2(j1)
                psi_j = (
                    self.c
                    * (j1 ** (-self.mu))
                    * np.cos(2 * np.pi * freq1 * x1)
                    * np.cos(2 * np.pi * freq2 * x2)
                )
                a_val += self.y[j] * psi_j
            values[0] = a_val

        def value_shape(self):
            return ()

    f = Constant(1.0)
    u = TrialFunction(V)
    v = TestFunction(V)

    a_expr = CoefficientA(y, c, mu, degree=3)
    a_form = dot(a_expr * grad(u), grad(v)) * dx

    u_sol = Function(V)
    solve(a_form == f * v * dx, u_sol, bc)

    G1 = len(x1_grid)
    G2 = len(x2_grid)
    U = np.zeros((G1, G2), dtype=np.float64)

    for i, x1 in enumerate(x1_grid):
        for j, x2 in enumerate(x2_grid):
            U[i, j] = u_sol(Point(float(x1), float(x2)))

    return U


torch.set_grad_enabled(False)

device = "cuda"
bs = 4
G = 100

currentdir = os.getcwd()
parentdir = os.path.dirname(currentdir)

testset_path = os.path.join(parentdir, "diff_pde_testset_10000.npy")
checkpoint_path = "checkpoints/tfno_diff_pde.pt"

y = np.load(testset_path).astype(np.float64)
N_eval, L = y.shape

x1_grid = np.linspace(0.0, 1.0, G)
x2_grid = np.linspace(0.0, 1.0, G)
X1, X2 = np.meshgrid(x1_grid, x2_grid, indexing="ij")

a_field = build_a_field_on_grid(y, X1, X2)
a_in = torch.from_numpy(a_field).float().unsqueeze(1).to(device)

u_ref_list = []
for n in range(N_eval):
    u_ref_list.append(solve_pde_u_on_grid(y[n, :], x1_grid, x2_grid, nx=50, ny=50))

u_ref = torch.from_numpy(np.stack(u_ref_list, axis=0)).float().unsqueeze(1).to(device)

from neuralop.models import TFNO

model = TFNO(
    n_modes=(16, 16),
    in_channels=1,
    out_channels=1,
    hidden_channels=64,
    n_layers=4,
).to(device)

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

errs = []

with torch.no_grad():
    for i in range(0, N_eval, bs):
        a_batch = a_in[i : i + bs]
        u_ref_b = u_ref[i : i + bs]

        u_pred_b = model(a_batch)

        err_b = torch.linalg.vector_norm(
            (u_ref_b - u_pred_b).reshape(u_ref_b.shape[0], -1), dim=1
        ) / torch.linalg.vector_norm(u_ref_b.reshape(u_ref_b.shape[0], -1), dim=1)

        errs.append(err_b.detach().cpu())

err = torch.cat(errs, dim=0).numpy()

plt.figure(figsize=(8, 2.5))
plt.boxplot(err, vert=False, widths=0.5, patch_artist=True)
plt.xscale("log")
plt.xlabel("Relative $L^2$ error (log scale)", fontsize=12)
plt.title(
    "Boxplot of relative $L^2$ Errors for the 2D parametric diffusion equation",
    fontsize=13,
)
plt.grid(True, axis="x", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
with open(csv_path, mode="a", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(new_data)